# Split Base File by Salesperson

Creates one workbook per salesperson from the Base File.

- Filters every sheet on the salesperson column ('Updated sales', or 'Sale person Name' on the USD sheet). Header = row 2, data from row 3.
- Keeps formatting, and re-points row formulas (e.g. `=T10*N10` -> `=T3*N3`) so they stay correct after rows move up.

## 1. Settings — edit these

In [9]:

OUTPUT_FOLDER = "split_output"

INPUT_FOLDER = "/home/isha/Desktop/sm-electronics/projects-aggregrator/"
FILE_NAME = "Base_File_Updation_North_Region_-Oct-26_working_file_-_Copy__2_.xlsx"

INPUT_FILE = INPUT_FOLDER + FILE_NAME
OUTPUT_FOLDER = INPUT_FOLDER + "split_output"

## 2. Imports and constants

In [10]:
import re
from copy import copy
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.formula.translate import Translator

SALES_COLS = {"updated sales", "sale person name"}
HEADER_ROW, FIRST_DATA_ROW = 2, 3

## 3. Helper functions

In [11]:
def norm(v):
    return str(v).strip().upper() if v not in (None, "") else None

def sales_col(ws):
    for c in ws[HEADER_ROW]:
        if c.value and str(c.value).strip().lower() in SALES_COLS:
            return c.column
    return None

def filter_sheet(ws, person):
    col = sales_col(ws)
    if not col:
        return
    target = FIRST_DATA_ROW
    for r in range(FIRST_DATA_ROW, ws.max_row + 1):
        if norm(ws.cell(r, col).value) != person:
            continue
        if r != target:  # move row r up to target, fixing formulas + styles
            for src in ws[r]:
                dst = ws.cell(target, src.column)
                v = src.value
                if isinstance(v, str) and v.startswith("="):
                    v = Translator(v, origin=src.coordinate).translate_formula(dst.coordinate)
                dst.value = v
                if src.has_style:
                    dst._style = copy(src._style)
            ws.row_dimensions[target].height = ws.row_dimensions[r].height
        target += 1
    if target <= ws.max_row:
        ws.delete_rows(target, ws.max_row - target + 1)

## 4. Find all salespersons

In [12]:
wb = load_workbook(INPUT_FILE)
people = sorted({norm(ws.cell(r, c).value)
                 for ws in wb if (c := sales_col(ws))
                 for r in range(FIRST_DATA_ROW, ws.max_row + 1)} - {None})
print(len(people), "salespersons found:")
for p in people:
    print(" -", p)

FileNotFoundError: [Errno 2] No such file or directory: '/home/isha/Desktop/sm-electronics/projects-aggregrator/Base_File_Updation_North_Region_-Oct-26_working_file_-_Copy__2_.xlsx'

## 5. Create one file per salesperson

In [ ]:
out = Path(OUTPUT_FOLDER); out.mkdir(exist_ok=True)
for person in people:
    wb = load_workbook(INPUT_FILE)          # fresh copy for each person
    for ws in wb:
        filter_sheet(ws, person)
    name = f"{Path(INPUT_FILE).stem}_{re.sub(r'[^A-Za-z0-9]+', '_', person).strip('_')}.xlsx"
    wb.save(out / name)
    print("Created", out / name)